In [2]:
import json, sqlite3, collections, pathlib
root = pathlib.Path(r'D:\code\DO272_Viewer')
seed = json.loads((root / 'python_backend' / 'seed_data' / 'airports.json').read_text(encoding='utf-8'))
print('Seed keys:', list(seed))
print('Seed airports:', [(a['icao'], a['source'], len(a['features'])) for a in seed.values()])
con = sqlite3.connect(f'file:{(root / "python_backend" / "do272_aerodromes.db").as_posix()}?mode=ro', uri=True)
print('Database:', con.execute('select icao, source from do272_aerodromes order by icao').fetchall())

Seed keys: ['KSFO', 'EGLL', 'EDDF', 'OMDB']
Seed airports: [('KSFO', 'Curated Hub (DO-272/ED-99)', 62), ('EGLL', 'Curated Hub (DO-272/ED-99)', 42), ('EDDF', 'Curated Hub (DO-272/ED-99)', 42), ('OMDB', 'Curated Hub (DO-272/ED-99)', 41)]
Database: [('EDDF', 'Curated Hub (DO-272/ED-99)'), ('EGLL', 'Curated Hub (DO-272/ED-99)'), ('KSFO', 'Curated Hub (DO-272/ED-99)'), ('OMDB', 'Curated Hub (DO-272/ED-99)'), ('VOBL', 'OpenStreetMap Aeroway Ingest')]


The seed contains four curated airports. SQLite additionally contains VOBL from an OSM ingest, so no curated VOBL counterpart is present in the seed or selected database.

In [3]:
rows = con.execute('select aerodrome_id, payload from do272_features order by aerodrome_id, feature_index').fetchall()
db_features = collections.defaultdict(list)
for icao, payload in rows:
    db_features[icao].append(json.loads(payload))
for icao in ['KSFO','EGLL','EDDF','OMDB','VOBL']:
    fs = db_features[icao]
    counts = collections.Counter(f.get('featureClass') for f in fs)
    print('\n', icao, len(fs), dict(counts))
    print('runway examples:', [(f.get('designator'), f.get('lengthMeters'), f.get('widthMeters'), len(f.get('coordinates') or [])) for f in fs if f.get('featureClass') == 'RunwayElement'][:10])
    print('missing class/id/coordinates:', {k:sum(not f.get(k) for f in fs) for k in ['featureClass','id','coordinates']})
print('\nVOBL metadata:', con.execute("select * from do272_aerodromes where icao='VOBL'").description, con.execute("select * from do272_aerodromes where icao='VOBL'").fetchone())


 KSFO 62 {'RunwayElement': 4, 'RunwayThreshold': 2, 'RunwayBlastPad': 1, 'TaxiwayElement': 3, 'TaxiwayGuidanceLine': 6, 'RunwayHoldingPosition': 2, 'ParkingStandArea': 3, 'StandGuidanceLine': 3, 'PassengerBoardingBridge': 1, 'HotSpot': 2, 'RunwaySafetyArea': 1, 'VerticalStructure': 2, 'AerodromeReferencePoint': 1, 'LocalizerAntenna': 1, 'GlidePathAntenna': 1, 'VisualGlideSlopeIndicator': 1, 'WindDirectionIndicator': 1, 'Stopway': 1, 'Clearway': 1, 'RunwayShoulder': 1, 'RunwayIntersection': 1, 'RunwayMarking': 1, 'RunwayCenterlinePoint': 1, 'TaxiwayIntersection': 1, 'TaxiwayShoulder': 1, 'IntermediateHoldingPosition': 1, 'TaxiwayHoldingPositionLight': 1, 'EnhancedTaxiwayCenterline': 1, 'TaxiwayMarking': 1, 'RapidExitTaxiwayIndicator': 1, 'ApronElement': 1, 'ApronShoulder': 1, 'ParkingStandLocation': 1, 'DeicingArea': 1, 'ServiceRoad': 1, 'PassengerWalkway': 1, 'RunwayEndSafetyArea': 1, 'ObjectFreeArea': 1, 'ObstacleFreeZone': 1, 'VerticalStructureLine': 1, 'VerticalStructurePolygon': 1

VOBL contains 340 stored OSM-derived objects, while the curated airports contain 41–62 features covering many more feature classes. Counts describe different representations, not verified completeness. VOBL's metadata includes placeholder-looking city, country and IATA values; the importer code must be consulted to establish their origin.

In [5]:
import urllib.request, urllib.parse, datetime
centers = {'KSFO':(37.619,-122.375),'EGLL':(51.470,-0.454),'EDDF':(50.038,8.562),'OMDB':(25.253,55.366)}
osm = {}
for icao, (lat, lon) in centers.items():
    box = ','.join(f'{x:.4f}' for x in (lat-.045,lon-.045,lat+.045,lon+.045))
    q = '[out:json][timeout:25];(' + ''.join(f'{kind}["aeroway"="{tag}"]({box});' for kind,tag in [('way','runway'),('way','taxiway'),('way','apron'),('node','parking_position'),('node','holding_position'),('node','control_tower'),('node','windsock')]) + ');out body geom;'
    data = urllib.parse.urlencode({'data':q}).encode()
    req = urllib.request.Request('https://overpass-api.de/api/interpreter', data=data, headers={'User-Agent':'DO272-Spatial-GIS/1.0','Content-Type':'application/x-www-form-urlencoded'})
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.load(resp)
    osm[icao] = result['elements']
    print(icao, 'OSM timestamp', result.get('osm3s',{}).get('timestamp_osm_base'), 'raw', dict(collections.Counter(e.get('tags',{}).get('aeroway') for e in osm[icao])))

KSFO OSM timestamp 2026-11-13T05:46:14Z raw {'holding_position': 86, 'parking_position': 2, 'windsock': 1, 'runway': 8, 'taxiway': 257, 'apron': 18}
EGLL OSM timestamp 2026-11-13T05:46:14Z raw {'holding_position': 288, 'parking_position': 70, 'windsock': 1, 'runway': 4, 'taxiway': 417, 'apron': 39}
EDDF OSM timestamp 2026-11-13T05:47:17Z raw {'holding_position': 348, 'windsock': 10, 'runway': 4, 'taxiway': 600, 'apron': 35}
OMDB OSM timestamp 2026-11-13T05:47:17Z raw {'holding_position': 235, 'parking_position': 28, 'apron': 219, 'taxiway': 375, 'runway': 5}


The live OSM results are bounding-box query results, not airport-boundary validated. The importer currently fetches apron, holding-position and windsock objects but does not convert them. Raw OSM counts therefore must not be interpreted as imported feature counts.

In [6]:
mapped = {'runway':'RunwayElement','taxiway':'TaxiwayGuidanceLine','parking_position':'ParkingStandArea','control_tower':'VerticalStructure'}
for icao, elems in osm.items():
    curated = db_features[icao]
    print('\n',icao,'curated total',len(curated),'OSM importable',sum(e.get('tags',{}).get('aeroway') in mapped for e in elems))
    print('curated runways:', [(f['designator'],f['lengthMeters'],f['widthMeters']) for f in curated if f['featureClass']=='RunwayElement'])
    runways = [e for e in elems if e.get('tags',{}).get('aeroway')=='runway']
    print('OSM runway ways (id, ref, length, width, surface, nodes):', [(e['id'],e['tags'].get('ref'),e['tags'].get('length'),e['tags'].get('width'),e['tags'].get('surface'),len(e.get('geometry',[]))) for e in runways])
    print('OSM missing runway tags:', {k:sum(not e['tags'].get(k) for e in runways) for k in ['ref','length','width','surface','pcn']})
print('\nVOBL stored runway source tags:', [(f['designator'], f['lengthMeters'], f['widthMeters'],f.get('properties',{}).get('osmTags',{})) for f in db_features['VOBL'] if f['featureClass']=='RunwayElement'])


 KSFO curated total 62 OSM importable 267
curated runways: [('28L/10R', 3469, 61), ('28R/10L', 3618, 61), ('01L/19R', 2332, 61), ('01R/19L', 2636, 61)]
OSM runway ways (id, ref, length, width, surface, nodes): [(22567191, '10L/28R', '3618', '61', 'asphalt', 32), (23365560, '10R/28L', '3231', '61', 'asphalt', 27), (510551655, '10L/28R', None, '61', 'asphalt', 4), (586154075, '01R/19L', '2636', '61', 'asphalt', 3), (586154076, '01R/19L', '2636', '61', 'asphalt', 23), (586154078, '01L/19R', '2286', '61', 'asphalt', 18), (1333615087, '01L/19R', '2286', '61', 'asphalt', 6), (1333615088, '10R/28L', '3231', '61', 'asphalt', 3)]
OSM missing runway tags: {'ref': 0, 'length': 1, 'width': 0, 'surface': 0, 'pcn': 8}

 EGLL curated total 42 OSM importable 491
curated runways: [('09L/27R', 3902, 50), ('09R/27L', 3660, 50)]
OSM runway ways (id, ref, length, width, surface, nodes): [(2798800, '09R/27L', '3660', '50', 'paved', 24), (2798802, '09L/27R', '3902', '50', 'paved', 27), (510279283, None, Non

### Interpretation and visual comparison

Counts below compare whole curated feature sets with OSM objects that the current importer would convert; classes and segmentation differ, so bar lengths do **not** measure accuracy or coverage. Each block represents approximately 20 objects.

| Airport | Curated | Importable live OSM | Comparison (curated 🟦 / OSM 🟧) |
|---|---:|---:|---|
| KSFO | 62 | 267 | 🟦🟦🟦 / 🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧 |
| EGLL | 42 | 491 | 🟦🟦 / 🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧 |
| EDDF | 42 | 604 | 🟦🟦 / 🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧 |
| OMDB | 41 | 408 | 🟦🟦 / 🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧 |
| VOBL | unavailable | 340 stored OSM | — / 🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧 |

The VOBL number is from the database, not the live queries. Across the four live airports, 21 runway ways lack PCN tags; the current converter supplies a default rather than preserving the missing value. EGLL also has two unnamed runway ways. Runway ways may be segments of one physical runway. Comparing matching designators exposes several differing length or width values but does not establish which is correct.

### Q&A
Both sources can be used together only with separate provenance, geometry-aware matching, conflict review and preservation of unknown attributes; the current ICAO-keyed upsert replaces the previous dataset. VOBL has no curated counterpart in the inspected database or seed, so no two-source comparison is possible for it.

### Data Analysis Key Findings
- KSFO's curated 28L/10R length is 3,469 m; OSM 10R/28L is 3,231 m. EGLL matching runway lengths agree. EDDF 07R/25L width is 60 m curated versus 45 m OSM. OMDB 12L/30R is 4,000 m curated versus 4,300 m OSM. These are disagreements, not validated errors.
- OSM returns runway segments, not necessarily one object per runway. OSM features include ways and nodes while the curated dataset has a broader class taxonomy.
- VOBL has 340 stored OSM features, including two runways, and no curated entry. Its stored metadata includes IATA `OBL` and generic city/country fields.

### Insights or Next Steps
- Verify runway attributes and all aeronautical uses against appropriately authoritative, current data; neither source has been validated as operationally authoritative.
- Preserve source snapshots, reconcile by designator and spatial geometry, display conflicts and source attribution, and avoid inventing missing safety-critical values.